# pandas 3/3 : `groupby`, `merge` et `concat`

**Objectifs** : calculer des statistiques **par groupe** avec `groupby`, puis combiner des tables avec `merge` (jointure) et `concat` (empilement).

*Durée estimée : 45 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/cdrutinuspro/intro-data-python.git
    %cd intro-data-python/notebooks

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/titanic.csv")
df.head(3)

## 1. `groupby` : séparer, appliquer, combiner

Le principe (*split-apply-combine*) :

1. **séparer** les lignes en groupes selon les valeurs d'une colonne
2. **appliquer** une fonction (moyenne, somme, comptage…) à chaque groupe
3. **combiner** les résultats dans un nouveau tableau

In [ ]:
df.groupby("Sex")["Survived"].mean()    # taux de survie par sexe

In [ ]:
df.groupby("Pclass")["Age"].mean()      # âge moyen par classe

On peut voir concrètement les groupes en les parcourant :

In [ ]:
for classe, sous_df in df.groupby("Pclass"):
    print(f"Classe {classe} : {len(sous_df)} passagers, tarif moyen {sous_df['Fare'].mean():.1f}")

### Plusieurs agrégations à la fois : `agg`

In [ ]:
df.groupby("Pclass")["Fare"].agg(["count", "mean", "median", "max"])

Avec des **agrégations nommées**, on contrôle le nom des colonnes de résultat et on peut agréger plusieurs colonnes différentes :

In [ ]:
df.groupby("Pclass").agg(
    nb_passagers=("PassengerId", "count"),
    age_moyen=("Age", "mean"),
    tarif_moyen=("Fare", "mean"),
    taux_survie=("Survived", "mean"),
)

### Grouper par plusieurs colonnes

In [ ]:
res = df.groupby(["Pclass", "Sex"])["Survived"].mean()
res

Le résultat a un **index à deux niveaux**. `unstack()` fait pivoter le dernier niveau en colonnes, ce qui donne un tableau plus lisible :

In [ ]:
res.unstack()

In [ ]:
# Même résultat avec pivot_table, souvent plus direct
df.pivot_table(index="Pclass", columns="Sex", values="Survived", aggfunc="mean")

### Récupérer un DataFrame « normal »

`reset_index()` transforme l'index de groupes en colonnes ordinaires (pratique pour la suite, notamment pour les graphiques).

In [ ]:
df.groupby("Embarked")["Survived"].mean().reset_index()

In [ ]:
# Et on peut trier le résultat
df.groupby("Embarked")["Fare"].mean().sort_values(ascending=False)

---
### ✏️ Exercices (`groupby`)

### ✏️ Exercice 1

Calculez le taux de survie selon le port d'embarquement (`Embarked`).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.groupby("Embarked")["Survived"].mean()
```

</details>

### ✏️ Exercice 2

Calculez l'âge moyen par classe **et** par sexe, sous forme d'un tableau avec les classes en lignes et les sexes en colonnes.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.pivot_table(index="Pclass", columns="Sex", values="Age", aggfunc="mean")
```

</details>

### ✏️ Exercice 3

Pour chaque classe, donnez le nombre de passagers, le tarif minimum et le tarif maximum (agrégations nommées).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.groupby("Pclass").agg(n=("PassengerId", "count"), fare_min=("Fare", "min"), fare_max=("Fare", "max"))
```

</details>

## 2. `merge` : joindre deux tables

On a souvent des informations réparties dans plusieurs tables, reliées par une **clé** commune. Ici, `Embarked` contient un code de port (`C`, `Q`, `S`) et un autre fichier décrit ces ports :

In [ ]:
ports = pd.read_csv("../data/ports.csv")
ports

In [ ]:
df_ports = df.merge(ports, on="Embarked")
df_ports[["Name", "Embarked", "Port", "Pays"]].head()

### Les types de jointure : `how`

- `inner` (défaut) : on ne garde que les clés présentes **dans les deux** tables
- `left` : on garde toutes les lignes de la table de gauche
- `right` : toutes celles de la table de droite
- `outer` : tout, des deux côtés

Observons sur de petites tables :

In [ ]:
etudiants = pd.DataFrame({"id": [1, 2, 3], "nom": ["Alice", "Bob", "Chloé"]})
notes = pd.DataFrame({"id": [2, 3, 4], "note": [14, 9, 16]})

print("--- inner"); print(etudiants.merge(notes, on="id", how="inner"))
print("--- left");  print(etudiants.merge(notes, on="id", how="left"))
print("--- right"); print(etudiants.merge(notes, on="id", how="right"))
print("--- outer"); print(etudiants.merge(notes, on="id", how="outer"))

Et sur nos données : deux passagers n'ont pas de port renseigné (`NaN`), et le port `B` (Belfast) n'a aucun passager.

In [ ]:
print("df           :", len(df))
print("inner        :", len(df.merge(ports, on="Embarked", how="inner")))
print("left         :", len(df.merge(ports, on="Embarked", how="left")))
print("outer        :", len(df.merge(ports, on="Embarked", how="outer")))

**Réflexe à avoir** : après un `merge`, vérifier le nombre de lignes. Pour enrichir une table sans perdre de lignes, on utilise en général `how="left"`.

### Clés de noms différents et `indicator`

Si la colonne clé n'a pas le même nom dans les deux tables : `left_on` / `right_on`. `indicator=True` ajoute une colonne qui indique d'où vient chaque ligne.

In [ ]:
ports2 = ports.rename(columns={"Embarked": "code_port"})
res = df.merge(ports2, left_on="Embarked", right_on="code_port", how="outer", indicator=True)
res["_merge"].value_counts()

### Exploiter la jointure : combiner merge et groupby

In [ ]:
df.merge(ports, on="Embarked", how="left").groupby("Pays")["Survived"].agg(["count", "mean"])

---
### ✏️ Exercices (`merge`)

### ✏️ Exercice 4

Quel est le tarif moyen payé par pays d'embarquement ? (jointure avec `ports`, puis `groupby`)

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.merge(ports, on="Embarked", how="left").groupby("Pays")["Fare"].mean()
```

</details>

### ✏️ Exercice 5

On dispose d'une table des classes : `classes = pd.DataFrame({"Pclass": [1, 2, 3], "Libelle": ["Première", "Seconde", "Troisième"]})`. Ajoutez le libellé à `df` et vérifiez que le nombre de lignes n'a pas changé.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
classes = pd.DataFrame({"Pclass": [1, 2, 3], "Libelle": ["Première", "Seconde", "Troisième"]})
d = df.merge(classes, on="Pclass", how="left")
print(len(d) == len(df))
print(d[["Name", "Libelle"]].head())
```

</details>

## 3. `concat` : empiler des tables

`pd.concat` colle des tables **les unes sous les autres** (`axis=0`, défaut) ou **côte à côte** (`axis=1`). Cas typique : plusieurs fichiers de même structure (un par mois, par année…) à regrouper.

In [ ]:
jan = pd.DataFrame({"produit": ["A", "B"], "ventes": [10, 20]})
fev = pd.DataFrame({"produit": ["A", "B"], "ventes": [12, 18]})

pd.concat([jan, fev])

L'index d'origine est conservé (d'où les doublons `0, 1, 0, 1`). Avec `ignore_index=True`, on renumérote ; avec `keys`, on garde l'origine de chaque bloc :

In [ ]:
print(pd.concat([jan, fev], ignore_index=True))
print()
print(pd.concat([jan, fev], keys=["janvier", "février"]))

### Colonnes différentes

Les colonnes sont alignées **par nom** ; ce qui manque est rempli par `NaN`.

In [ ]:
a = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
b = pd.DataFrame({"x": [5], "z": [6]})
pd.concat([a, b], ignore_index=True)

### Côte à côte : `axis=1`

Les lignes sont alignées **par index**.

In [ ]:
ages = df["Age"].head(3)
noms = df["Name"].head(3)
pd.concat([noms, ages], axis=1)

### Exemple complet : séparer puis recombiner

In [ ]:
hommes = df[df["Sex"] == "male"]
femmes = df[df["Sex"] == "female"]

tout = pd.concat([hommes, femmes])
print(hommes.shape, femmes.shape, tout.shape)
print(tout.sort_index().equals(df))

## Récapitulatif

| Je veux… | Code |
|---|---|
| statistique par groupe | `df.groupby("col")["val"].mean()` |
| plusieurs statistiques | `.agg(["mean", "max"])` ou `.agg(nom=("col", "fonction"))` |
| tableau croisé | `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)` |
| joindre deux tables | `df1.merge(df2, on="clé", how="left")` |
| empiler des lignes | `pd.concat([df1, df2], ignore_index=True)` |
| coller des colonnes | `pd.concat([df1, df2], axis=1)` |

---
## Exercices (`concat`)

### ✏️ Exercice 6

Découpez `df` en trois DataFrames selon la classe (`Pclass` = 1, 2, 3), puis recombinez-les avec `concat`. Vérifiez que le nombre de lignes est identique à celui de `df`.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
c1 = df[df["Pclass"] == 1]
c2 = df[df["Pclass"] == 2]
c3 = df[df["Pclass"] == 3]
recombine = pd.concat([c1, c2, c3], ignore_index=True)
print(len(recombine) == len(df))
```

</details>

### ✏️ Exercice 7

**Défi** : pour chaque classe, trouvez les 2 passagers ayant payé le tarif le plus élevé. *Indice : triez par tarif décroissant, puis `groupby("Pclass").head(2)`.*

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
df.sort_values("Fare", ascending=False).groupby("Pclass").head(2).sort_values(["Pclass", "Fare"], ascending=[True, False])[["Pclass", "Name", "Fare"]]
```

</details>